# Reconstruction Impact: Partition Change

This notebook quantifies **ST-unit Expression-Space Change** for observation-paired spatial units. It runs the P1CRC VisiumHD sp-SVC and P2CRC Xenium Fibroblast sc-SVC routes from their post-analysis YAML files.

**Evidence boundary.** A changed partition describes a representation change under the declared preprocessing and resolution rule. It does not establish biological mechanism, truth, or clinical relevance.

In [ ]:
import os
import sys
from pathlib import Path

def find_repo_root(start):
    for candidate in (start, *start.parents):
        if (candidate / 'pyproject.toml').is_file():
            return candidate
    raise FileNotFoundError('Could not locate the REVISE repository root.')

REPO_ROOT = find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from revise.analysis.reconstruction_impact import (
    file_sha256,
    load_reconstruction_impact_config,
    run_partition_analysis,
    write_analysis_artifacts,
)

CONFIGS = {
    'visiumhd': REPO_ROOT / 'configs/analysis/reconstruction_impact_visiumhd_p1crc.yaml',
    'xenium_fibroblast': REPO_ROOT / 'configs/analysis/reconstruction_impact_xenium_p2crc_fibroblast.yaml',
}
CONFIG_PATH = Path(os.environ.get('REVISE_RECONSTRUCTION_IMPACT_CONFIG', CONFIGS['visiumhd']))
config = load_reconstruction_impact_config(CONFIG_PATH)
output_root = Path(os.environ.get('REVISE_ANALYSIS_OUTPUT_ROOT', REPO_ROOT / config['output']['dir']))
print({'config': str(CONFIG_PATH), 'output_root': str(output_root), 'available_routes': list(CONFIGS)})

In [ ]:
comparison_config = config['partition_change']['comparisons'][0]
raw_path = REPO_ROOT / comparison_config['raw_h5ad']
recon_path = REPO_ROOT / comparison_config['reconstructed_spatial_h5ad']
raw_full = ad.read_h5ad(raw_path)
recon_full = ad.read_h5ad(recon_path)

# The reconstructed spatial carrier declares the cohort. Raw is indexed by exactly
# those IDs after an explicit subset check; no silent set intersection is used.
if not recon_full.obs_names.isin(raw_full.obs_names).all():
    raise ValueError('The reconstructed spatial IDs are not all present in Raw.')
raw = raw_full[recon_full.obs_names].copy()
recon = recon_full.copy()
sample_n = config['partition_change'].get('sample_n_units')
if sample_n and raw.n_obs > sample_n:
    rng = np.random.default_rng(config['partition_change']['random_state'])
    sampled_ids = np.sort(rng.choice(raw.obs_names.to_numpy(), size=sample_n, replace=False))
    raw = raw[sampled_ids].copy()
    recon = recon[sampled_ids].copy()

input_audit = pd.DataFrame([{
    'sample_id': config['sample']['id'], 'input_role': 'raw', 'path': str(raw_path),
    'n_obs': raw.n_obs, 'n_vars': raw.n_vars, 'obs_names_unique': raw.obs_names.is_unique,
    'paired_obs_count': raw.n_obs, 'paired_obs_fraction': 1.0, 'status': 'ok',
}, {
    'sample_id': config['sample']['id'], 'input_role': 'reconstructed_spatial', 'path': str(recon_path),
    'n_obs': recon.n_obs, 'n_vars': recon.n_vars, 'obs_names_unique': recon.obs_names.is_unique,
    'paired_obs_count': recon.n_obs, 'paired_obs_fraction': 1.0, 'status': 'ok',
}])
input_audit

In [ ]:
analysis = run_partition_analysis(
    raw, recon,
    level1_col=comparison_config['level1_column'],
    final_cluster_key=comparison_config.get('reconstructed_cluster_key'),
    resolution_mode=config['partition_change']['mode'],
    resolution_candidates=config['partition_change']['level1_resolution_candidates'],
    within_level1_resolution=config['partition_change']['within_level1_resolution'],
    random_state=config['partition_change']['random_state'],
    n_top_genes=config['partition_change']['n_top_genes'],
)
summary = pd.concat([result.summary for result in analysis.comparisons.values()], ignore_index=True)
summary['sample_id'] = config['sample']['id']
summary['resolution'] = analysis.resolution
summary['resolution_source'] = analysis.resolution_source
summary['n_shared_genes'] = analysis.audit['n_shared_genes']
summary

In [ ]:
manifest = {
    'sample_id': config['sample']['id'], 'analysis': 'reconstruction_impact_partition_change',
    'resolution': analysis.resolution, 'resolution_source': analysis.resolution_source,
    'pairing_audit': analysis.audit, 'config_path': str(CONFIG_PATH),
    'sampling': {'n_units': raw.n_obs, 'seed': config['partition_change']['random_state']},
    'inputs': {
        'raw': {'path': str(raw_path), 'sha256': file_sha256(raw_path), 'n_obs_full': raw_full.n_obs},
        'reconstructed_spatial': {'path': str(recon_path), 'sha256': file_sha256(recon_path), 'n_obs_full': recon_full.n_obs},
    },
}
write_analysis_artifacts(output_root, config=config, manifest=manifest, input_audit=input_audit)
st_unit_dir = output_root / 'st_unit'
figure_dir = st_unit_dir / 'figures'
figure_dir.mkdir(parents=True, exist_ok=True)
summary.to_csv(st_unit_dir / 'partition_summary.csv', index=False)
analysis.sweep.to_csv(st_unit_dir / 'raw_level1_resolution_sweep.csv', index=False)
for edge, result in analysis.comparisons.items():
    result.mapping.to_csv(st_unit_dir / f'cluster_mapping__{edge}.csv', index=False)
    result.contingency.to_csv(st_unit_dir / f'contingency__{edge}.csv')
    result.assignments.to_csv(st_unit_dir / f'unit_change__{edge}.csv.gz', index=False, compression='gzip')

ax = summary.set_index('comparison_edge')[['st_unit_change_fraction', 'balanced_cluster_change']].plot.bar(rot=30)
ax.set_ylabel('change fraction')
ax.set_ylim(0, 1)
ax.figure.tight_layout()
ax.figure.savefig(figure_dir / 'partition_change_summary.png', dpi=180)
plt.show()

for edge, result in analysis.comparisons.items():
    fig, ax = plt.subplots(figsize=(5, 4))
    image = ax.imshow(result.contingency.to_numpy(), cmap='Blues')
    ax.set_xticks(range(result.contingency.shape[1]), result.contingency.columns, rotation=45, ha='right')
    ax.set_yticks(range(result.contingency.shape[0]), result.contingency.index)
    ax.set_xlabel('reconstructed cluster')
    ax.set_ylabel('raw cluster')
    ax.set_title(edge)
    fig.colorbar(image, ax=ax, label='paired units')
    fig.tight_layout()
    fig.savefig(figure_dir / f'cluster_overlap__{edge}.png', dpi=180)
    plt.show()